# Reusable Template — Univariate Time-Series Diagnostic Pipeline

Copy this notebook as the starting point for a new time-series EDA project. It packages the diagnostic workflow (mean/variance/ergodicity checks → white-noise test → ACF/PACF → stationarity tests → transforms → optional cross-correlation) into small, reusable functions you call on any new series.

**To reuse:** update the `CONFIG` cell, point `load_data()` at your file, then run all cells. Everything downstream reads from `CONFIG` and the loaded `series`.


## 0. Config — edit this for each new project

In [ ]:
CONFIG = {
    "data_path": "airline-passengers.csv",   # TODO: path to your CSV
    "date_col_is_index": True,                # True if dates are the index column (index_col=0)
    "date_format": "%Y-%m",                   # TODO: match your date strings, or None to let pandas infer
    "value_col": "Passengers",                # TODO: name of the column to analyze
    "seasonal_period": 12,                    # TODO: e.g. 12 for monthly/yearly, 7 for daily/weekly, None if unknown
    "acf_lags": 50,
    "ljung_box_lags": [10, 20, 50],
    "alpha": 0.05,
}


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.signal import correlate

plt.rcParams['figure.figsize'] = (10, 4)


## 2. Load data

In [ ]:
def load_data(cfg):
    if cfg["date_col_is_index"]:
        df = pd.read_csv(cfg["data_path"], header=0, index_col=0)
        df.index = pd.to_datetime(df.index, format=cfg["date_format"]) if cfg["date_format"] else pd.to_datetime(df.index)
    else:
        df = pd.read_csv(cfg["data_path"])
    return df

df = load_data(CONFIG)
series = df[CONFIG["value_col"]].astype(float)
series.plot(title=f'{CONFIG["value_col"]} — raw series')
plt.show()
series.head()


## 3. Descriptive diagnostics: mean, variance, rolling stats, seasonal boxplot

In [ ]:
def describe_series(series, seasonal_period=None):
    print(f"n = {len(series)}")
    print(f"mean = {series.mean():.3f}, std = {series.std():.3f}")
    half = len(series) // 2
    print(f"first-half mean/std = {series[:half].mean():.3f} / {series[:half].std():.3f}")
    print(f"second-half mean/std = {series[half:].mean():.3f} / {series[half:].std():.3f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(series)
    ax[0].plot(series.rolling(window=max(2, len(series)//20)).mean(), color='orange', label='rolling mean')
    ax[0].plot(series.rolling(window=max(2, len(series)//20)).std(), color='green', label='rolling std')
    ax[0].legend(); ax[0].set_title('Series with rolling mean/std')

    if seasonal_period and hasattr(series.index, 'year'):
        sns.boxplot(x=series.index.year, y=series.values, ax=ax[1])
        ax[1].set_title('Distribution by year')
    else:
        ax[1].hist(series, bins=30)
        ax[1].set_title('Histogram')
    plt.tight_layout(); plt.show()

describe_series(series, seasonal_period=CONFIG["seasonal_period"])


## 4. Stationarity test suite (ADF + KPSS)

In [ ]:
def stationarity_report(series, name="series", alpha=0.05):
    s = pd.Series(series).dropna()
    adf_stat, adf_p, *_ = adfuller(s, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(s, regression='c', nlags='auto')
    adf_verdict = 'stationary' if adf_p < alpha else 'NON-stationary'
    kpss_verdict = 'NON-stationary' if kpss_p < alpha else 'stationary'
    print(f"--- {name} ---")
    print(f"ADF : stat={adf_stat:.3f} p={adf_p:.4f} -> {adf_verdict}")
    print(f"KPSS: stat={kpss_stat:.3f} p={kpss_p:.4f} -> {kpss_verdict}")
    return {"adf_p": adf_p, "kpss_p": kpss_p, "adf_verdict": adf_verdict, "kpss_verdict": kpss_verdict}

_ = stationarity_report(series, "raw series", alpha=CONFIG["alpha"])


## 5. ACF / PACF plots

In [ ]:
def plot_acf_pacf(series, lags=50, alpha=0.05, title_prefix=""):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(series, lags=lags, alpha=alpha, ax=ax[0]); ax[0].set_title(f"{title_prefix} ACF")
    plot_pacf(series, lags=lags, alpha=alpha, ax=ax[1]); ax[1].set_title(f"{title_prefix} PACF")
    plt.tight_layout(); plt.show()

plot_acf_pacf(series, lags=CONFIG["acf_lags"], alpha=CONFIG["alpha"], title_prefix="Raw")


## 6. White-noise check (Ljung-Box)

In [ ]:
def white_noise_check(series, lags):
    result = acorr_ljungbox(series, lags=lags, return_df=True)
    print(result)
    verdict = "white noise (fail to reject H0)" if (result['lb_pvalue'] > 0.05).all() else "NOT white noise (reject H0 at >=1 lag)"
    print("Verdict:", verdict)
    return result

_ = white_noise_check(series.dropna(), CONFIG["ljung_box_lags"])


## 7. Transform pipeline: try differencing / seasonal differencing / log until stationary

In [ ]:
def try_transforms(series, seasonal_period=None, alpha=0.05):
    """Runs a small ladder of standard transforms and reports stationarity after each.
    Returns a dict of {label: transformed_series} for further inspection/plotting."""
    candidates = {"raw": series}

    first_diff = series.diff(1).dropna()
    candidates["first_diff"] = first_diff

    if seasonal_period:
        seasonal_diff = series.diff(seasonal_period).dropna()
        candidates["seasonal_diff"] = seasonal_diff
        seasonal_then_first = seasonal_diff.diff(1).dropna()
        candidates["seasonal_diff+first_diff"] = seasonal_then_first

    if (series > 0).all():
        log_series = np.log(series)
        candidates["log"] = log_series
        candidates["log+first_diff"] = log_series.diff(1).dropna()
        if seasonal_period:
            candidates["log+seasonal_diff"] = log_series.diff(seasonal_period).dropna()
            candidates["log+seasonal_diff+first_diff"] = log_series.diff(seasonal_period).diff(1).dropna()

    print(f"{'label':30s} {'ADF p':>10s} {'KPSS p':>10s}  verdict")
    for label, s in candidates.items():
        adf_p = adfuller(s, autolag='AIC')[1]
        kpss_p = kpss(s, regression='c', nlags='auto')[1]
        both_stationary = (adf_p < alpha) and (kpss_p >= alpha)
        flag = "  <-- both tests agree: stationary" if both_stationary else ""
        print(f"{label:30s} {adf_p:10.4f} {kpss_p:10.4f}{flag}")

    return candidates

transformed = try_transforms(series, seasonal_period=CONFIG["seasonal_period"], alpha=CONFIG["alpha"])


In [ ]:
# Inspect the ACF/PACF of whichever transform looked stationary above.
# TODO: set this to the winning key from the printout, e.g. "log+seasonal_diff+first_diff"
BEST_TRANSFORM = "first_diff"

plot_acf_pacf(transformed[BEST_TRANSFORM], lags=CONFIG["acf_lags"], alpha=CONFIG["alpha"],
              title_prefix=BEST_TRANSFORM)
_ = white_noise_check(transformed[BEST_TRANSFORM], CONFIG["ljung_box_lags"])


## 8. Seasonal decomposition (optional, if a seasonal period is known)

In [ ]:
if CONFIG["seasonal_period"]:
    decomp = seasonal_decompose(series, model='additive', period=CONFIG["seasonal_period"])
    decomp.plot()
    plt.show()


## 9. Cross-correlation utility (for a second series, e.g. an external driver)

In [ ]:
def plot_ccf(data_a, data_b, lag_lookback, percentile=95):
    n = len(data_a)
    ccf = correlate(data_a - np.mean(data_a), data_b - np.mean(data_b), method='direct') / (
        np.std(data_a) * np.std(data_b) * n
    )
    _min = (len(ccf) - 1) // 2 - lag_lookback
    _max = (len(ccf) - 1) // 2 + (lag_lookback - 1)
    z = {90: 1.645, 95: 1.96, 99: 2.576}[percentile] / np.sqrt(n)

    plt.figure(figsize=(12, 4))
    plt.stem(np.arange(-lag_lookback, lag_lookback - 1), ccf[_min:_max])
    plt.axhline(z, color='b', ls='--'); plt.axhline(-z, color='b', ls='--')
    plt.axvline(0, color='black')
    plt.title('Cross-Correlation'); plt.xlabel('Lag'); plt.ylabel('Correlation')
    plt.show()
    return ccf

# Example usage once you have a second series `other_series` aligned to the same index:
# plot_ccf(transformed[BEST_TRANSFORM], other_series_transformed, lag_lookback=30)


## 10. Summary cell — fill in your conclusions

- Chosen transform: `BEST_TRANSFORM = ...`
- Stationary? (ADF/KPSS agreement): ...
- Seasonal period detected/used: ...
- Candidate AR order (from PACF): ...
- Candidate MA order (from ACF): ...
- Any leading/lagging external series found via CCF: ...
- Next step: proceed to AR/MA/ARIMA/SARIMA model fitting on the transformed series.
